In [10]:
import pandas as pd
import json

# Volta uma pasta (../), entra em src e lê o dicionário
with open('/home/bruno/UESC/DataSUS/DadosTUB/src/dicionario_sinan_tuberculose.json', 'r', encoding='utf-8') as f:
    dicionario = json.load(f)

# Volta uma pasta (../), entra em data/processed e lê o CSV
df = pd.read_csv('/home/bruno/UESC/DataSUS/DadosTUB/data/processed/tuberculose_ilheus_itabuna_limpo_2014_2024.csv')

In [11]:
import pandas as pd
import numpy as np

# 1. CONVERSÃO DE DATAS
# Procuramos dinamicamente todas as colunas que começam com 'Data' ou contêm 'Data' no nome
colunas_de_data = [col for col in df.columns if 'Data' in col]

print("A converter datas para formato datetime...")
for col in colunas_de_data:
    # O errors='coerce' é fundamental aqui. Se houver uma data inválida digitada 
    # pelo enfermeiro (ex: 99999999), o Pandas converte para NaT (Not a Time) sem quebrar o código.
    df[col] = pd.to_datetime(df[col], format='%Y%m%d', errors='coerce')


# 2. ENGENHARIA DA IDADE (DESCODIFICAÇÃO DO SINAN)
def descodificar_idade(codigo):
    if pd.isna(codigo):
        return np.nan
    
    codigo = str(codigo).strip()
    if len(codigo) != 4:
        return np.nan
        
    tipo = codigo[0]
    valor = int(codigo[1:])
    
    if tipo == '4':   # 4 = Idade em Anos
        return valor
    elif tipo == '3': # 3 = Idade em Meses (Menor de 1 ano = 0 anos)
        return 0
    elif tipo == '2': # 2 = Idade em Dias (Menor de 1 ano = 0 anos)
        return 0
    elif tipo == '5': # 5 = Idade acima de 100 anos (valor já é em anos)
        return valor
    else:
        return np.nan

print("A extrair a idade real em anos...")
if 'Idade_Codigo' in df.columns:
    df['Idade_Anos'] = df['Idade_Codigo'].apply(descodificar_idade)
    df.drop(columns=['Idade_Codigo'], inplace=True) # Removemos a coluna antiga para limpar a base


# 3. MAPEAMENTO DE CATEGORIAS (DE-PARA)
print("A aplicar rótulos clínicos às variáveis categóricas...")

mapa_sexo = {'M': 'Masculino', 'F': 'Feminino', 'I': 'Ignorado'}

mapa_encerramento = {
    '1': 'Cura', '2': 'Abandono', '3': 'Óbito por TB', 
    '4': 'Óbito por outras causas', '5': 'Transferência', 
    '6': 'Mudança de Diagnóstico', '7': 'TB-DR', 
    '8': 'Mudança de Esquema', '9': 'Falência', '10': 'Abandono Primário'
}

mapa_entrada = {
    '1': 'Caso Novo', '2': 'Recidiva', '3': 'Reingresso após abandono', 
    '4': 'Não sabe', '5': 'Transferência', '6': 'Pós-óbito'
}

# Aplicamos os mapas apenas se as colunas sobreviveram ao nosso corte de 45% anterior
if 'Sexo' in df.columns:
    df['Sexo'] = df['Sexo'].map(mapa_sexo).fillna(df['Sexo'])
    
if 'Situacao_Encerramento_Final' in df.columns:
    df['Situacao_Encerramento_Final'] = df['Situacao_Encerramento_Final'].map(mapa_encerramento).fillna(df['Situacao_Encerramento_Final'])
    
if 'Tipo_Entrada' in df.columns:
    df['Tipo_Entrada'] = df['Tipo_Entrada'].map(mapa_entrada).fillna(df['Tipo_Entrada'])

print("\n✅ Formatação e Tipagem concluídas com sucesso!")

# Exibimos a nova estrutura de tipos de dados para auditoria
df[['Data_Notificacao', 'Idade_Anos', 'Sexo', 'Situacao_Encerramento_Final']].head()

A converter datas para formato datetime...
A extrair a idade real em anos...
A aplicar rótulos clínicos às variáveis categóricas...

✅ Formatação e Tipagem concluídas com sucesso!


/tmp/ipykernel_4366/3045248873.py:66: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['Situacao_Encerramento_Final'] = df['Situacao_Encerramento_Final'].map(mapa_encerramento).fillna(df['Situacao_Encerramento_Final'])
/tmp/ipykernel_4366/3045248873.py:69: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['Tipo_Entrada'] = df['Tipo_Entrada'].map(mapa_entrada).fillna(df['Tipo_Entrada'])


,Data_Notificacao,Idade_Anos,Sexo,Situacao_Encerramento_Final
0,2015-12-04,57,Feminino,1.0
1,2015-12-04,70,Feminino,1.0
2,2015-12-23,34,Masculino,1.0
3,2015-12-07,54,Masculino,1.0
4,2015-12-07,30,Masculino,1.0


In [12]:
# 1. IDENTIFICAÇÃO DE DUPLICATAS EXATAS
# O Pandas varre o dataset e marca como 'True' qualquer linha que seja 
# uma cópia 100% exata de uma linha anterior.
total_linhas_antes = len(df)
duplicatas_exatas = df.duplicated().sum()

print("--- PILAR 3: ANÁLISE DE UNICIDADE (MÉTODO EXATO) ---")
print(f"Total de registos (pacientes) antes da limpeza: {total_linhas_antes}")
print(f"⚠️ Linhas 100% duplicadas encontradas: {duplicatas_exatas}")

# 2. REMOÇÃO AUTOMÁTICA
if duplicatas_exatas > 0:
    print("\nA remover duplicatas exatas do sistema...")
    # keep='first' (padrão) mantém a primeira ocorrência e apaga as cópias
    df.drop_duplicates(inplace=True)
    
    # Reinicia o índice para garantir que não ficam "buracos" na numeração das linhas
    df.reset_index(drop=True, inplace=True)
    
    print(f"✅ Limpeza concluída! Nova volumetria: {len(df)} registos únicos.")
else:
    print("\n✅ Excelentes notícias: A base não possui erros de duplicação exata!")

# Exibe uma amostra para garantir que a estrutura continua íntegra
display(df.head(3))

--- PILAR 3: ANÁLISE DE UNICIDADE (MÉTODO EXATO) ---
Total de registos (pacientes) antes da limpeza: 3431
⚠️ Linhas 100% duplicadas encontradas: 0

✅ Excelentes notícias: A base não possui erros de duplicação exata!


,Tipo_Notificacao,Agravo_Codigo,Data_Notificacao,Ano_Notificacao,UF_Notificacao,Municipio_Notificacao,Regional_Notificacao,Data_Diagnostico,Ano_Nascimento,Sexo,...,Agravo_Drogas_Ilicitas,Agravo_Tabagismo,Teste_Rapido_Molecular,Teste_Sensibilidade,Uso_Antiretroviral,Baciloscopia_Apos_6_Meses,Transferencia_Confirmada,UF_Transferencia,Municipio_Transferencia,Idade_Anos
0,2,A169,2015-12-04,2015,29,291360,1385,2015-12-03,1958.0,Feminino,...,2.0,2.0,5.0,7.0,0.0,0.0,0.0,NaN,NaN,57
1,2,A169,2015-12-04,2015,29,291360,1385,2015-12-04,1945.0,Feminino,...,2.0,2.0,5.0,5.0,0.0,0.0,0.0,NaN,NaN,70
2,2,A169,2015-12-23,2015,29,291480,1386,2015-12-05,1981.0,Masculino,...,2.0,1.0,5.0,0.0,0.0,0.0,0.0,NaN,NaN,34


In [13]:
# 1. DEFINIÇÃO DA CHAVE SINTÉTICA (IMPRESSÃO DIGITAL DO PACIENTE)
# Estas colunas formam o nosso critério rigoroso de semelhança
colunas_chave_sintetica = [
    'Idade_Anos', 
    'Sexo', 
    'Raca_Cor', 
    'Municipio_Residencia', 
    'Data_Diagnostico'
]

# 2. BUSCA DE DUPLICATAS SUSPEITAS
# keep=False garante que o Pandas traga TANTO a linha original quanto a cópia 
# para podermos comparar as duas lado a lado.
df_suspeitos = df[df.duplicated(subset=colunas_chave_sintetica, keep=False)].copy()

print("--- ANÁLISE DE UNICIDADE AVANÇADA (CHAVE SINTÉTICA) ---")
print(f"Total de casos suspeitos de duplicação: {len(df_suspeitos)}")

# 3. EXIBIÇÃO PARA AUDITORIA (SE HOUVER SUSPEITOS)
if len(df_suspeitos) > 0:
    print("\n⚠️ Atenção: Encontramos pacientes com a mesma 'Impressão Digital'!")
    
    # Ordenamos pelas colunas da chave para que as cópias fiquem coladas uma na outra
    df_suspeitos.sort_values(by=colunas_chave_sintetica, inplace=True)
    
    # Selecionamos apenas colunas úteis para a sua inspeção visual
    colunas_auditoria = colunas_chave_sintetica + [
        'Data_Notificacao', 'Tipo_Entrada', 'Situacao_Encerramento_Final'
    ]
    
    # Exibe os primeiros 10 casos suspeitos
    display(df_suspeitos[colunas_auditoria])
    
    print("\n💡 DICA CLÍNICA: Olhe para o 'Tipo_Entrada'. Se as duas linhas dizem 'Caso Novo',")
    print("é um erro de duplicação do sistema. Se uma diz 'Caso Novo' e a outra meses/anos")
    print("depois diz 'Recidiva', então NÃO é duplicata, é o mesmo paciente que adoeceu de novo.")
else:
    print("\n✅ Incrível! A base passou no teste rigoroso. Nenhuma duplicata suspeita encontrada.")

--- ANÁLISE DE UNICIDADE AVANÇADA (CHAVE SINTÉTICA) ---
Total de casos suspeitos de duplicação: 74

⚠️ Atenção: Encontramos pacientes com a mesma 'Impressão Digital'!


,Idade_Anos,Sexo,Raca_Cor,Municipio_Residencia,Data_Diagnostico,Data_Notificacao,Tipo_Entrada,Situacao_Encerramento_Final
1125,19,Feminino,2.0,291480,2014-03-23,2014-04-23,1,2.0
1126,19,Feminino,2.0,291480,2014-03-23,2014-07-25,3,2.0
2118,19,Masculino,4.0,291480,2024-04-17,2024-04-17,1,2.0
2119,19,Masculino,4.0,291480,2024-04-17,2024-08-02,3,2.0
3317,19,Masculino,9.0,291480,2017-02-14,2017-02-16,1,1.0
...,...,...,...,...,...,...,...,...
1935,56,Feminino,2.0,291480,2020-09-17,2021-02-04,3,1.0
2467,57,Masculino,2.0,291480,2021-01-25,2021-02-04,1,2.0
2468,57,Masculino,2.0,291480,2021-01-25,2022-08-19,6,3.0
3158,61,Masculino,4.0,291360,2017-05-16,2017-05-16,1,1.0



💡 DICA CLÍNICA: Olhe para o 'Tipo_Entrada'. Se as duas linhas dizem 'Caso Novo',
é um erro de duplicação do sistema. Se uma diz 'Caso Novo' e a outra meses/anos
depois diz 'Recidiva', então NÃO é duplicata, é o mesmo paciente que adoeceu de novo.


In [14]:
# 1. AUDITORIA CRONOLÓGICA (VIAGEM NO TEMPO)
# Verifica se a data de encerramento é menor (mais antiga) que o diagnóstico
mascara_erro_cronologico = df['Data_Encerramento'] < df['Data_Diagnostico']
erros_cronologicos = df[mascara_erro_cronologico]

# 2. AUDITORIA DE TEMPO DE TRATAMENTO (OUTLIERS)
# Calcula a diferença em dias entre diagnóstico e encerramento
df['Dias_Ate_Encerramento'] = (df['Data_Encerramento'] - df['Data_Diagnostico']).dt.days

# Consideramos suspeito quem encerrou no mesmo dia (0 dias) sem ser óbito/mudança de diagnóstico,
# ou tratamentos absurdamente longos (ex: > 5 anos / 1825 dias)
mascara_tempo_suspeito = (df['Dias_Ate_Encerramento'] < 0) | (df['Dias_Ate_Encerramento'] > 1825)
erros_tempo = df[mascara_tempo_suspeito & df['Data_Encerramento'].notna()]

print("--- PILAR 4: VALIDAÇÃO DE REGRAS DE NEGÓCIO ---")
print(f"⚠️ Inconsistências Cronológicas (Encerramento antes do Diagnóstico): {len(erros_cronologicos)} casos")
print(f"⚠️ Tempos de Tratamento Suspeitos (> 5 anos ou negativos): {len(erros_tempo)} casos")

if len(erros_cronologicos) > 0 or len(erros_tempo) > 0:
    print("\nVisualizando uma amostra das inconsistências:")
    colunas_vis = ['Data_Diagnostico', 'Data_Encerramento', 'Dias_Ate_Encerramento', 'Situacao_Encerramento_Final']
    
    # Junta os dois tipos de erros para exibir
    df_erros = pd.concat([erros_cronologicos, erros_tempo]).drop_duplicates()
    display(df_erros[colunas_vis].head(10))
else:
    print("\n✅ Excelente! Nenhuma quebra de regra cronológica encontrada.")

--- PILAR 4: VALIDAÇÃO DE REGRAS DE NEGÓCIO ---
⚠️ Inconsistências Cronológicas (Encerramento antes do Diagnóstico): 1 casos
⚠️ Tempos de Tratamento Suspeitos (> 5 anos ou negativos): 1 casos

Visualizando uma amostra das inconsistências:


,Data_Diagnostico,Data_Encerramento,Dias_Ate_Encerramento,Situacao_Encerramento_Final
151,2015-08-05,2015-07-21,NaN,2.0
151,2015-08-05,2015-07-21,-15.0,2.0


In [15]:
print(f"Volumetria antes do filtro: {len(df)} pacientes")

# O til (~) significa "NÃO". Ou seja, mantenha no dataframe apenas quem 
# NÃO tem erro cronológico e NÃO tem tempo de tratamento suspeito.
df = df[~mascara_erro_cronologico & ~mascara_tempo_suspeito].copy()
df.reset_index(drop=True, inplace=True)

print(f"Volumetria após o filtro: {len(df)} pacientes")
print("\n✅ CERTIFICADO DE QUALIDADE: Base de dados 100% validada e pronta!")

Volumetria antes do filtro: 3431 pacientes
Volumetria após o filtro: 3430 pacientes

✅ CERTIFICADO DE QUALIDADE: Base de dados 100% validada e pronta!


In [17]:
import os

# 1. CRIAR A ESTRUTURA DE PASTAS
os.makedirs('data/processed', exist_ok=True)
os.makedirs('src', exist_ok=True)

# 2. GUARDAR O DATASET FINAL
caminho_dados = 'data/processed/tuberculose_ilheus_itabuna_final.csv'
df.to_csv(caminho_dados, index=False, encoding='utf-8')

# 3. PULAR O DICIONÁRIO (Opcional por enquanto)
# caminho_dict = 'src/dicionario_sinan_tuberculose.json'
# with open(caminho_dict, 'w', encoding='utf-8') as f:
#     json.dump(dicionario_sinan_completo, f, ensure_ascii=False, indent=4)

print(f"✅ Arquivo CSV salvo com sucesso em: {caminho_dados}")
print("Você já pode abrir o Notebook 02 para a análise!")

✅ Arquivo CSV salvo com sucesso em: data/processed/tuberculose_ilheus_itabuna_final.csv
Você já pode abrir o Notebook 02 para a análise!
